# Getting Started

This chapter runs a complete GENESIS analysis workflow in Python, end to end, on
the **BPTI** protein-in-water system bundled with genepie's test data. In about
ten lines you will:

1. load a molecular structure,
2. read a trajectory,
3. compute a C&alpha; RMSD, and
4. plot the result with Plotly.

Everything here executes on the bundled data, so you can reproduce it immediately
after installing genepie.


In [ ]:
import numpy as np
from genepie import genesis_exe, SMolecule
from genepie.tests.conftest import BPTI_PDB, BPTI_PSF, BPTI_DCD

# The reference structure (ref=) is what RMSD is measured against.
mol = SMolecule.from_file(pdb=BPTI_PDB, psf=BPTI_PSF, ref=BPTI_PDB)
print(f"Loaded {mol.num_atoms} atoms, {mol.num_residues} residues")

## Load the trajectory

`crd_convert` reads one or more trajectory files and returns a **tuple**:
`(list_of_trajectories, subset_molecule)`. There is one `STrajectories` object
per input file, and `subset_molecule` is the molecule reduced to the atoms picked
by `selection` (here `"all"`, so it matches the input).

```{admonition} API note
:class: important
`crd_convert` returns a *tuple*. Unpack it as `trajs, subset_mol = crd_convert(...)`.
Indexing the first trajectory is `trajs[0]`.
```


In [ ]:
trajs, subset_mol = genesis_exe.crd_convert(
    mol,
    trj_files=[str(BPTI_DCD)],
    trj_format="DCD",
    trj_type="COOR+BOX",
    selection="all",
)
traj = trajs[0]
print(f"{len(trajs)} trajectory, shape (nframe, natom, 3) = {traj.coords.shape}")

## Compute a C&alpha; RMSD

`rmsd_analysis` takes the molecule (for masses and the reference coordinates) and
a trajectory. Selections are ordinary strings in GENESIS syntax, and
`fitting_method` is case-insensitive.


In [ ]:
result = genesis_exe.rmsd_analysis(
    mol,
    traj,
    analysis_selection="an:CA",
    fitting_selection="an:CA",
    fitting_method="TR+ROT",
)
print(type(result.rmsd), result.rmsd.shape)
print(f"RMSD: mean={result.rmsd.mean():.3f} A, max={result.rmsd.max():.3f} A")

## Plot the result

`result.rmsd` is a NumPy array, so it drops straight into any plotting or ML library.

In [ ]:
import plotly.io as pio
import plotly.graph_objects as go
pio.renderers.default = "notebook"

fig = go.Figure()
fig.add_trace(go.Scatter(
    y=result.rmsd, mode="lines+markers", name="C&alpha; RMSD",
    line=dict(color="#4C72B0", width=2.5),
    marker=dict(size=8, color="#4C72B0", line=dict(width=1, color="white")),
    fill="tozeroy", fillcolor="rgba(76,114,176,0.12)",
))
fig.update_layout(
    title=dict(text="<b>BPTI C&alpha; RMSD vs. reference</b>", font=dict(size=18)),
    xaxis_title="Frame",
    yaxis_title="RMSD (&#8491;)",
    template="plotly_white",
    font=dict(family="Inter, Helvetica, Arial, sans-serif", size=13, color="#333"),
    hovermode="x unified",
    margin=dict(l=60, r=30, t=60, b=50),
    height=400,
)
fig

## Where to go next

- **[Molecules](tutorials/01_molecule.ipynb)** — inspect and visualize structures.
- **[Trajectories](tutorials/02_trajectory.ipynb)** — selections, fitting, centering.
- **[Lazy loading](tutorials/02b_lazy_loading.ipynb)** — analyze trajectories too big for RAM.
- **[Reference](reference/api.md)** — the full analysis API.
